# Formation Value Optimizer — Playground #1

Build the most **premium-heavy** starting XI within a **$100** budget.

**Model**
- Squad = 15 (standard FPL): 2 GK, 5 DEF, 5 MID, 3 FWD.
- Each position has 4 hardcoded price buckets: `b4 = premium` … `b1 = warmer`.
- A *formation* (e.g. 3-5-2) chooses the starting outfield players; the rest bench.
- GK: 1 premium starts, 1 warmer benches. Bench outfielders are forced to warmers (b1).
- Objective: maximize the sum of starting-XI bucket levels, tie-broken by cheapest spend.

Edit the **Config** cell, then *Run All*.

## Config — edit me

In [311]:
BUDGET = 100.0
SCORE_MIN = 21              # return every affordable setup scoring >= this

# N price buckets per position (any N works — derived from PRICES below).
# index 0 = b1 (warmer, cheapest) ... last index = premium (most expensive).
PRICES = {
    "GK":   [4.0, 5.5, 5.5, 6.0, 6.0],    # 5.0 - 6.0
    "DEF":  [4.5, 5.5, 5.5, 6.5, 8.0],    # 5.0 - 8.0
    "MID":  [4.5, 6.5, 7.5, 8.0, 12.5],   # 5.5 - 11.5
    "FWD":  [4.5, 6.5, 7.5, 8.0, 15.5],   # 6.0 - 15.5
}

# Minimum-quality rule per starting line, as a list of (bucket_group, min_count).
# Each entry means: "at least `min_count` starters whose bucket is in bucket_group".
# Buckets are 1-based: 1 = b1 (warmer) ... N = premium.
#   - singleton group  -> per-bucket minimum, e.g. ({5}, 1)   = >=1 b5 (premium)
#   - multi group (OR) -> "strong" style,      e.g. ({4,5},1) = >=1 b4-or-b5
#   - stack several     -> combine, e.g. [({4,5},2), ({5},1)] = >=2 strong AND >=1 premium
MIN_BY_GROUP = {
    "DEF": [({4,5}, 3)],
    "MID": [({4,5}, 3)],
    "FWD": [({5}, 1)],
}

# max b1 warmers allowed among the STARTING XI of each line (bench not counted)
MAX_START_WARMERS = {"DEF": 1, "MID": 0, "FWD": 0}

# squad shape (total players bought per position)
SQUAD = {"GK": 2, "DEF": 5, "MID": 5, "FWD": 3}

# optional: weight a premium level differently per position (1.0 = neutral).
# e.g. bump FWD/MID if attacking premiums matter more than defensive ones.
POS_WEIGHT = {"DEF": 1.0, "MID": 1.0, "FWD": 1.0}

FORMATIONS = [
    (3, 5, 2), (3, 4, 3), (4, 5, 1), (4, 4, 2),
    (4, 3, 3),
]

## Solver

In [312]:
from itertools import product

# number of buckets is inferred from the price table (assumes all lines match)
NBUCKETS = len(PRICES["DEF"])
PREMIUM = NBUCKETS - 1   # index of the premium (most expensive) bucket
WARMER = 0              # index of the warmer (cheapest) bucket


def bucket_options(n):
    """All bucket-count splits (c1..cN) for n starters across NBUCKETS buckets."""
    return [c for c in product(range(n + 1), repeat=NBUCKETS) if sum(c) == n]


def _meets_min_groups(pos, combo_line):
    """True if the line satisfies every (bucket_group, min_count) in MIN_BY_GROUP[pos]."""
    for group, min_count in MIN_BY_GROUP[pos]:
        if sum(combo_line[b - 1] for b in group) < min_count:  # buckets are 1-based
            return False
    return True


def _fixed_and_left(d_start, m_start, f_start):
    """Fixed cost (GK + forced bench warmers) and remaining budget."""
    gk_cost = PRICES["GK"][PREMIUM] + PRICES["GK"][WARMER]  # 1 premium starts + 1 warmer benches
    bench = {"DEF": SQUAD["DEF"] - d_start,
             "MID": SQUAD["MID"] - m_start,
             "FWD": SQUAD["FWD"] - f_start}
    bench_cost = sum(PRICES[p][WARMER] * n for p, n in bench.items())
    fixed = gk_cost + bench_cost
    return fixed, BUDGET - fixed


def all_for_formation(d_start, m_start, f_start, score_min=0.0):
    """Every affordable setup for a formation with score >= score_min.

    Rules on the STARTING XI (bench players are always forced warmers):
      * each line satisfies its MIN_BY_GROUP[pos] constraints (GK premium guaranteed).
      * <= MAX_START_WARMERS[pos] b1 warmers among the starters of each line.

    Returns a list of dict(combo, spend, score); combo maps position -> (c1..cN).
    """
    fixed, budget_left = _fixed_and_left(d_start, m_start, f_start)
    starts = {"DEF": d_start, "MID": m_start, "FWD": f_start}
    positions = ["DEF", "MID", "FWD"]
    # per line: meets min-by-group constraints, and few enough b1 warmers
    options = {p: [c for c in bucket_options(starts[p])
                   if _meets_min_groups(p, c) and c[WARMER] <= MAX_START_WARMERS[p]]
               for p in positions}

    out = []
    for cd in options["DEF"]:
        for cm in options["MID"]:
            for cf in options["FWD"]:
                combo = {"DEF": cd, "MID": cm, "FWD": cf}
                cost, score = 0.0, 0.0
                for p in positions:
                    for i, cnt in enumerate(combo[p]):
                        cost += cnt * PRICES[p][i]
                        score += cnt * (i + 1) * POS_WEIGHT[p]  # bucket level = index + 1
                if cost <= budget_left + 1e-9 and score >= score_min - 1e-9:
                    out.append({"combo": combo, "spend": cost + fixed, "score": score})
    return out


def best_for_formation(d_start, m_start, f_start):
    """Best (highest score, then cheapest) affordable setup, or None."""
    setups = all_for_formation(d_start, m_start, f_start)
    if not setups:
        return None
    return max(setups, key=lambda s: (round(s["score"], 6), -s["spend"]))


def bucket_name(i):
    """Label for bucket index i: b1 is the warmer, the last is the premium."""
    if i == WARMER:
        return "b1(warmer)"
    if i == PREMIUM:
        return f"b{NBUCKETS}(premium)"
    return f"b{i + 1}"


def fmt_combo(combo_line):
    parts = [f"{cnt}x {bucket_name(i)}" for i, cnt in enumerate(combo_line) if cnt]
    return ", ".join(parts) if parts else "-"

## Results

In [313]:
import pandas as pd

COLUMNS = ["formation", "score", "spend", "DEF", "MID", "FWD"]

rows = []
for f in FORMATIONS:
    # infeasible formations simply yield no setups (e.g. a MID min > MID starters)
    for s in all_for_formation(*f, score_min=SCORE_MIN):
        combo = s["combo"]
        rows.append({
            "formation": "-".join(map(str, f)),
            "score": round(s["score"], 2),
            "spend": round(s["spend"], 1),
            "DEF": fmt_combo(combo["DEF"]),
            "MID": fmt_combo(combo["MID"]),
            "FWD": fmt_combo(combo["FWD"]),
        })

df = pd.DataFrame(rows, columns=COLUMNS)
if len(df):
    df = df.sort_values(["score", "spend"], ascending=[False, True]).reset_index(drop=True)

print(f"All setups scoring >= {SCORE_MIN}  ->  {len(df)} found")
if not len(df):
    print("(no setups meet the current rules/budget — try relaxing MIN_BY_GROUP or SCORE_MIN)")
print(f"Starting XI also includes GK 1x {bucket_name(PREMIUM)}. "
      f"Bench = 1 GK + outfield warmers.\n")
df

All setups scoring >= 21  ->  3 found
Starting XI also includes GK 1x b5(premium). Bench = 1 GK + outfield warmers.



,formation,score,spend,DEF,MID,FWD
0,4-5-1,34.0,100.0,"1x b1(warmer), 3x b4","2x b2, 3x b4",1x b5(premium)
1,4-4-2,34.0,100.0,"1x b1(warmer), 3x b4","1x b2, 3x b4","1x b2, 1x b5(premium)"
2,4-3-3,34.0,100.0,"1x b1(warmer), 3x b4",3x b4,"2x b2, 1x b5(premium)"


## (formation, score) counts

How many qualifying setups share each formation + score, highest score first.

In [314]:
if len(df):
    counts = (df.groupby(["formation", "score"]).size()
                .reset_index(name="count")
                .sort_values(["score", "count"], ascending=[False, False])
                .reset_index(drop=True))
    print(f"{len(counts)} (formation, score) pairs across {len(df)} setups")
else:
    counts = pd.DataFrame(columns=["formation", "score", "count"])
    print("no setups to count")
counts

3 (formation, score) pairs across 3 setups


,formation,score,count
0,4-3-3,34.0,1
1,4-4-2,34.0,1
2,4-5-1,34.0,1


## Interactive sliders (optional)

Requires `ipywidgets`. Drag premium prices and watch the ranking update live.

In [315]:
import ipywidgets as widgets
from IPython.display import display

_FORM_LABELS = ["All"] + ["-".join(map(str, f)) for f in FORMATIONS]


def rerun(formation, def_prem, mid_prem, fwd_prem, score_min):
    # sliders tune each line's PREMIUM (top) bucket price
    PRICES["DEF"][PREMIUM] = def_prem
    PRICES["MID"][PREMIUM] = mid_prem
    PRICES["FWD"][PREMIUM] = fwd_prem
    chosen = FORMATIONS if formation == "All" else [tuple(map(int, formation.split("-")))]
    rows = []
    for f in chosen:
        for s in all_for_formation(*f, score_min=score_min):
            combo = s["combo"]
            rows.append({"formation": "-".join(map(str, f)), "score": round(s["score"], 2),
                         "spend": round(s["spend"], 1),
                         "DEF": fmt_combo(combo["DEF"]), "MID": fmt_combo(combo["MID"]),
                         "FWD": fmt_combo(combo["FWD"])})
    if not rows:
        print(f"No setups for {formation} scoring >= {score_min}")
        return
    out = pd.DataFrame(rows).sort_values(["score", "spend"], ascending=[False, True]).reset_index(drop=True)
    print(f"{len(out)} setups | formation={formation} | score >= {score_min}")
    display(out.head(50))


def _prem_slider(pos):
    base = PRICES[pos][PREMIUM]
    return widgets.FloatSlider(value=base, min=base - 4, max=base + 4, step=0.5,
                              description=f"{pos} b{NBUCKETS}", continuous_update=False)


# continuous_update=False -> callback fires once on release, not on every drag step
widgets.interact(
    rerun,
    formation=widgets.Dropdown(options=_FORM_LABELS, value="All", description="Formation"),
    def_prem=_prem_slider("DEF"),
    mid_prem=_prem_slider("MID"),
    fwd_prem=_prem_slider("FWD"),
    score_min=widgets.IntSlider(value=SCORE_MIN, min=10, max=41, step=1, description="score >=", continuous_update=False),
);

interactive(children=(Dropdown(description='Formation', options=('All', '3-5-2', '3-4-3', '4-5-1', '4-4-2', '4…